[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/02_conditional_probability_and_bayes/exercises.ipynb)

# Exercises — Module 02: Conditional Probability and Bayes' Theorem

20 fully solved problems in four tiers: L0 Concept Checks (4), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (4). Every numeric answer is recomputed by the code cell that follows its solution.

In [1]:
import numpy as np
from math import comb, log, exp
from scipy.special import digamma

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

def approx(a, b, tol=1e-9):
    """Assert two numbers agree, and report the pair."""
    assert abs(a - b) < tol, (a, b)
    return a

## L0 — Concept Checks

### Problem L0.1 — Conditioning as Renormalization

**Statement.** A fair die is rolled. Given that the outcome is even, what is the probability it is a 6? Explain the answer as a restriction-and-renormalization of the sample space.

**Intuition.** Knowing "even" throws away three of the six faces; the three survivors were equally likely before and stay equally likely after, so each carries one third.

**Solution.**

*Step 1.* Let $A = \{6\}$ and $B = \{2, 4, 6\}$, so $P(A \cap B) = P(\{6\}) = 1/6$ and $P(B) = 3/6$.

*Step 2.* Apply the definition of conditional probability:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{1/6}{3/6} = \frac{1}{3}.
$$

*Step 3.* Read the mechanism off the formula: conditioning discards $\{1,3,5\}$, and the surviving mass $1/2$ is rescaled by $1/P(B) = 2$, so each of the three survivors carries $1/3$.

$$
\boxed{P(6 \mid \text{even}) = \frac{1}{3}}
$$

**Key takeaway.** Conditioning is "restrict to $B$, then renormalize"; the relative proportions of outcomes inside $B$ never change.

In [2]:
faces = np.arange(1, 7)
p = np.full(6, 1/6)
B = faces % 2 == 0
A = faces == 6
exact = p[A & B].sum() / p[B].sum()
rolls = rng.integers(1, 7, 400_000)
empirical = np.mean(rolls[rolls % 2 == 0] == 6)
print(f"exact P(6 | even) = {exact:.6f} = 1/3 = {1/3:.6f}")
print(f"simulated (400k rolls, {np.sum(rolls % 2 == 0)} kept) = {empirical:.4f}")
approx(exact, 1/3)
assert abs(empirical - 1/3) < 0.01

exact P(6 | even) = 0.333333 = 1/3 = 0.333333
simulated (400k rolls, 199064 kept) = 0.3336


### Problem L0.2 — Disjoint Is Not Independent

**Statement.** Events $A$ and $B$ satisfy $A \cap B = \emptyset$ with $P(A) = 0.3$ and $P(B) = 0.4$. Are they independent? What is $P(A \mid B)$?

**Intuition.** Independence means learning $B$ tells you nothing about $A$; disjointness means learning $B$ tells you everything — that $A$ did not happen.

**Solution.**

*Step 1.* Independence would require $P(A \cap B) = P(A)P(B) = 0.3 \times 0.4 = 0.12$.

*Step 2.* Disjointness forces $P(A \cap B) = P(\emptyset) = 0 \ne 0.12$, so the events are **not** independent.

*Step 3.* Compute the conditional probability directly:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{0}{0.4} = 0 \ne 0.3 = P(A).
$$

$$
\boxed{\text{Not independent: } P(A \mid B) = 0 \ne P(A) = 0.3}
$$

**Key takeaway.** Mutually exclusive events with positive probabilities are perfectly negatively dependent, never independent.

In [3]:
PA, PB, PAB = 0.3, 0.4, 0.0
print(f"P(A)P(B) = {PA * PB:.2f}   P(A and B) = {PAB:.2f}   equal? {PA * PB == PAB}")
print(f"P(A | B) = {PAB / PB:.2f}   P(A) = {PA:.2f}")
assert PA * PB != PAB and PAB / PB == 0.0

P(A)P(B) = 0.12   P(A and B) = 0.00   equal? False
P(A | B) = 0.00   P(A) = 0.30


### Problem L0.3 — Transposed Conditionals

**Statement.** In a certain city, $P(\text{tests positive} \mid \text{has condition}) = 0.99$. A prosecutor argues this means $P(\text{has condition} \mid \text{tests positive}) = 0.99$. Identify the fallacy and state the exact relationship between the two quantities.

**Intuition.** The two conditionals divide the same joint probability by different denominators, so they agree only when those denominators agree.

**Solution.**

*Step 1.* Name the fallacy: transposing the conditional, the *prosecutor's fallacy*. Bayes' theorem gives the exact bridge,

$$
P(C \mid +) = P(+ \mid C) \cdot \frac{P(C)}{P(+)},
$$

so the two directions coincide exactly when $P(C) = P(+)$.

*Step 2.* Quantify with a rare condition, $P(C) = 10^{-3}$, and a false-positive rate of $1\%$:

$$
P(+) = 0.99 \times 0.001 + 0.01 \times 0.999 = 0.00099 + 0.00999 = 0.01098.
$$

*Step 3.* Invert:

$$
P(C \mid +) = \frac{0.00099}{0.01098} \approx 0.0902,
$$

an order of magnitude below $0.99$.

$$
\boxed{P(C \mid +) = P(+ \mid C)\,\frac{P(C)}{P(+)}; \text{ equality requires } P(C) = P(+)}
$$

**Key takeaway.** Reversing a conditional multiplies by the ratio of marginals — with rare events, that ratio is tiny.

In [4]:
sens, fpr, prev = 0.99, 0.01, 1e-3
p_plus = sens * prev + fpr * (1 - prev)
post = sens * prev / p_plus
print(f"P(+) = {p_plus:.5f}   P(C | +) = {post:.4f}   P(+ | C) = {sens:.2f}")
print(f"ratio of marginals P(C)/P(+) = {prev / p_plus:.4f}")
approx(post, sens * prev / p_plus)
assert abs(post - 0.0902) < 5e-4

P(+) = 0.01098   P(C | +) = 0.0902   P(+ | C) = 0.99
ratio of marginals P(C)/P(+) = 0.0911


### Problem L0.4 — Does Conditioning Preserve the Probability Rules?

**Statement.** Your team computed $P(A \mid B) = 0.5$ and $P(A^c \mid B) = 0.6$ from the same data pipeline. Without further information, prove there must be a bug.

**Intuition.** A fixed conditioning event turns $P(\cdot \mid B)$ into an ordinary probability measure, and every ordinary measure obeys the complement rule.

**Solution.**

*Step 1.* For fixed $B$ with $P(B) \gt 0$, the map $Q(\cdot) = P(\cdot \mid B)$ is a probability measure: it is non-negative, $Q(\Omega) = 1$, and countably additive because the pieces $A_i \cap B$ inherit disjointness.

*Step 2.* Therefore every axiom-derived identity holds for $Q$, in particular the complement rule

$$
P(A \mid B) + P(A^c \mid B) = Q(A) + Q(A^c) = Q(\Omega) = 1 .
$$

*Step 3.* The reported values give $0.5 + 0.6 = 1.1 \ne 1$, impossible for any single conditioning event $B$ — the two numbers must have been conditioned on different events, or one is miscomputed.

$$
\boxed{P(A \mid B) + P(A^c \mid B) = 1 \text{ always; } 0.5 + 0.6 = 1.1 \text{ is inconsistent}}
$$

**Key takeaway.** Conditional probabilities obey every axiom-derived identity; a violation is diagnostic of a bug, not of an unusual dataset.

In [5]:
# any joint table gives complement-rule-consistent conditionals; the reported pair cannot
joint = rng.dirichlet(np.ones(4)).reshape(2, 2)      # rows = A / A^c, cols = B / B^c
qA, qAc = joint[0, 0] / joint[:, 0].sum(), joint[1, 0] / joint[:, 0].sum()
print(f"random table: P(A|B) = {qA:.4f}, P(A^c|B) = {qAc:.4f}, sum = {qA + qAc:.12f}")
print(f"reported pair: 0.5 + 0.6 = {0.5 + 0.6}")
approx(qA + qAc, 1.0, 1e-12)
assert abs(0.5 + 0.6 - 1.0) > 1e-9

random table: P(A|B) = 0.2069, P(A^c|B) = 0.7931, sum = 1.000000000000
reported pair: 0.5 + 0.6 = 1.1


## L1 — Foundations

### Problem L1.1 — Two Cards Without Replacement

**Statement.** Two cards are drawn without replacement from a 52-card deck. (a) What is the probability both are aces? (b) Given the first is an ace, what is the probability the second is an ace?

**Intuition.** After an ace leaves the deck, both the number of aces and the deck size shrink by one, so the second draw lives on a smaller, poorer sample space.

**Solution.**

*Step 1.* Answer (b) first, because the chain rule needs it. Conditioning on $A_1$ leaves $51$ cards of which $3$ are aces, so

$$
P(A_2 \mid A_1) = \frac{3}{51} = \frac{1}{17}.
$$

*Step 2.* Apply the chain rule for (a):

$$
P(A_1 \cap A_2) = P(A_1)\,P(A_2 \mid A_1) = \frac{4}{52} \cdot \frac{3}{51} = \frac{12}{2652} = \frac{1}{221} \approx 0.004525 .
$$

*Step 3.* Cross-check by counting unordered pairs: $\binom{4}{2}/\binom{52}{2} = 6/1326 = 1/221$. Both routes agree.

$$
\boxed{P(\text{both aces}) = \frac{1}{221}, \qquad P(A_2 \mid A_1) = \frac{1}{17}}
$$

**Key takeaway.** Sampling without replacement is inherently sequential; the chain rule turns it into a product of easy one-step conditionals.

In [6]:
chain = (4/52) * (3/51)
counting = comb(4, 2) / comb(52, 2)
print(f"chain rule    = {chain:.8f}")
print(f"counting      = {counting:.8f}")
print(f"1/221         = {1/221:.8f}   P(A2|A1) = 3/51 = {3/51:.8f} = 1/17 = {1/17:.8f}")
approx(chain, 1/221, 1e-15); approx(counting, 1/221, 1e-15); approx(3/51, 1/17, 1e-15)

chain rule    = 0.00452489
counting      = 0.00452489
1/221         = 0.00452489   P(A2|A1) = 3/51 = 0.05882353 = 1/17 = 0.05882353


0.058823529411764705

### Problem L1.2 — Law of Total Probability with Two Factories

**Statement.** Factory 1 produces 60% of all units with a 2% defect rate; Factory 2 produces 40% with a 5% defect rate. (a) What fraction of all units is defective? (b) Given a unit is defective, what is the probability it came from Factory 2?

**Intuition.** The overall defect rate is a production-weighted average of the two rates; Bayes then re-attributes the observed defects back to the factories in proportion to how many each contributes.

**Solution.**

*Step 1.* The factories partition the units, so the law of total probability applies:

$$
P(D) = P(D \mid F_1)P(F_1) + P(D \mid F_2)P(F_2) = 0.02 \times 0.6 + 0.05 \times 0.4 = 0.012 + 0.020 = 0.032 .
$$

*Step 2.* Apply Bayes' theorem with the same partition:

$$
P(F_2 \mid D) = \frac{P(D \mid F_2)P(F_2)}{P(D)} = \frac{0.020}{0.032} = 0.625 .
$$

*Step 3.* Interpret: Factory 2 makes only $40\%$ of units but accounts for $62.5\%$ of defects, because its rate is $2.5$ times higher.

$$
\boxed{P(D) = 0.032, \qquad P(F_2 \mid D) = 0.625}
$$

**Key takeaway.** Total probability averages conditional rates by source weights; Bayes re-attributes observed effects back to sources.

In [7]:
w = np.array([0.6, 0.4])          # production shares
d = np.array([0.02, 0.05])        # defect rates
PD = float(w @ d)
post = d * w / PD
print(f"P(D) = {PD:.4f}   posterior over factories = {post}")
approx(PD, 0.032, 1e-12); approx(post[1], 0.625, 1e-12)

P(D) = 0.0320   posterior over factories = [0.375 0.625]


np.float64(0.6250000000000001)

### Problem L1.3 — Monty Hall from the Mechanism

**Statement.** You pick door 1 of 3; a car is behind one uniformly random door. The host, who knows the car's location, opens a *goat* door among $\{2,3\}$, choosing uniformly when both hide goats. He opens door 3. Compute $P(\text{car behind 2} \mid \text{host opens 3})$ from first principles.

**Intuition.** The host's options depend on where the car is, so which door he opens is evidence; "car behind 2" forces his hand, "car behind 1" leaves him a coin flip.

**Solution.**

*Step 1.* Set up priors and likelihoods. With $C_i$ = "car behind door $i$" and $H_3$ = "host opens door 3", each $P(C_i) = 1/3$ and the mechanism gives

$$
P(H_3 \mid C_1) = \tfrac12, \qquad P(H_3 \mid C_2) = 1, \qquad P(H_3 \mid C_3) = 0 .
$$

*Step 2.* Compute the evidence by total probability over the partition $\{C_1, C_2, C_3\}$:

$$
P(H_3) = \tfrac12 \cdot \tfrac13 + 1 \cdot \tfrac13 + 0 \cdot \tfrac13 = \tfrac12 .
$$

*Step 3.* Apply Bayes' theorem:

$$
P(C_2 \mid H_3) = \frac{1 \times \frac13}{\frac12} = \frac{2}{3}, \qquad P(C_1 \mid H_3) = \frac{\frac12 \times \frac13}{\frac12} = \frac{1}{3}.
$$

Switching doubles the win probability; the asymmetry lies entirely in the likelihoods.

$$
\boxed{P(\text{switch wins}) = \frac{2}{3}}
$$

**Key takeaway.** Condition on the full generating mechanism of the observation (who could have done what), not merely on the revealed fact.

In [8]:
prior = np.full(3, 1/3)
lik = np.array([0.5, 1.0, 0.0])            # P(host opens 3 | car behind i)
post = prior * lik / (prior * lik).sum()
print(f"P(H3) = {(prior * lik).sum():.4f}   posterior = {post}")

n = 300_000                                 # simulate the full mechanism, then filter
car = rng.integers(0, 3, n)
pick = rng.integers(0, 3, n)
coin = rng.integers(0, 2, n)
host = np.where(car == pick,
                np.where(coin == 0, (pick + 1) % 3, (pick + 2) % 3),
                3 - car - pick)
keep = (pick == 0) & (host == 2)
print(f"simulated P(car behind 2 | pick 1, host opens 3) = {(car[keep] == 1).mean():.4f}"
      f"   ({keep.sum()} runs kept)")
approx(post[1], 2/3, 1e-12)
assert abs((car[keep] == 1).mean() - 2/3) < 0.01

P(H3) = 0.5000   posterior = [0.3333 0.6667 0.    ]
simulated P(car behind 2 | pick 1, host opens 3) = 0.6672   (49803 runs kept)


### Problem L1.4 — Pairwise but Not Mutually Independent

**Statement.** Toss two fair coins. Let $A$ = "first is heads", $B$ = "second is heads", $C$ = "the two results agree". Show that $A, B, C$ are pairwise independent but not mutually independent.

**Intuition.** Any two of the three events leave the third genuinely uncertain, but all three together are logically linked: $C$ is a deterministic function of $A$ and $B$.

**Solution.**

*Step 1.* On the uniform space $\{HH, HT, TH, TT\}$ we have $A = \{HH, HT\}$, $B = \{HH, TH\}$, $C = \{HH, TT\}$, each of probability $1/2$.

*Step 2.* Check the three pairs: $A \cap B = A \cap C = B \cap C = \{HH\}$, each of probability $1/4 = \frac12 \cdot \frac12$. All pairs satisfy the product rule, so the events are pairwise independent.

*Step 3.* Check the triple: $A \cap B \cap C = \{HH\}$, so

$$
P(A \cap B \cap C) = \frac14 \ne \frac18 = P(A)P(B)P(C).
$$

*Step 4.* Explain the failure: $C$ occurs exactly when $A$ and $B$ agree, so $P(C \mid A \cap B) = 1$ — the third event carries no independent randomness once the first two are known.

$$
\boxed{\text{Pairwise independent, yet } P(A \cap B \cap C) = \tfrac14 \ne \tfrac18}
$$

**Key takeaway.** Mutual independence must be checked on every sub-collection; pairwise checks miss higher-order structure (Bernstein's construction).

In [9]:
outcomes = ["HH", "HT", "TH", "TT"]
p = np.full(4, 0.25)
A = np.array([o[0] == "H" for o in outcomes])
B = np.array([o[1] == "H" for o in outcomes])
C = np.array([o[0] == o[1] for o in outcomes])
P = lambda mask: float(p[mask].sum())
print(f"P(A)={P(A):.2f} P(B)={P(B):.2f} P(C)={P(C):.2f}")
print(f"pairs: {P(A & B):.2f} {P(A & C):.2f} {P(B & C):.2f}   (each should be 0.25)")
print(f"triple: {P(A & B & C):.2f}   product: {P(A) * P(B) * P(C):.3f}")
for pair in [(A, B), (A, C), (B, C)]:
    approx(P(pair[0] & pair[1]), P(pair[0]) * P(pair[1]), 1e-12)
assert abs(P(A & B & C) - P(A) * P(B) * P(C)) > 0.1

P(A)=0.50 P(B)=0.50 P(C)=0.50
pairs: 0.25 0.25 0.25   (each should be 0.25)
triple: 0.25   product: 0.125


### Problem L1.5 — Odds-Form Sequential Update

**Statement.** A coin is either fair ($P(H) = 0.5$) or biased ($P(H) = 0.8$), with prior odds 1:1. You observe the sequence H, H, T, H. Compute the posterior odds and probability that the coin is biased.

**Intuition.** Each flip is an independent piece of evidence, so its likelihood ratio multiplies into the running odds; heads favour the biased coin, tails favour the fair one.

**Solution.**

*Step 1.* Compute the per-flip Bayes factors (biased versus fair): heads gives $0.8/0.5 = 1.6$, tails gives $0.2/0.5 = 0.4$.

*Step 2.* Multiply, since flips are independent given the coin type:

$$
\text{posterior odds} = 1 \times 1.6 \times 1.6 \times 0.4 \times 1.6 = 1.6384 .
$$

*Step 3.* Convert odds $o$ to probability $o/(1+o)$:

$$
P(\text{biased} \mid \text{HHTH}) = \frac{1.6384}{2.6384} \approx 0.6210 .
$$

*Step 4.* Check in log-odds, where the evidence adds: $3\ln 1.6 + \ln 0.4 = 1.4100 - 0.9163 = 0.4937$, and $\sigma(0.4937) \approx 0.6210$.

$$
\boxed{P(\text{biased} \mid \text{data}) \approx 0.6210}
$$

**Key takeaway.** Sequential Bayes is multiplicative in odds and additive in log-odds; each observation contributes an independent evidence increment.

In [10]:
data = [1, 1, 0, 1]                       # 1 = heads
bf = np.where(np.array(data) == 1, 0.8/0.5, 0.2/0.5)
odds = float(np.prod(bf))                 # prior odds 1
p_biased = odds / (1 + odds)
logit = float(np.log(bf).sum())
print(f"per-flip Bayes factors = {bf}")
print(f"posterior odds = {odds:.4f}   P(biased) = {p_biased:.4f}")
print(f"log-odds = {logit:.4f}   sigma(log-odds) = {1/(1 + np.exp(-logit)):.4f}")
direct = (0.5 * 0.8**3 * 0.2) / (0.5 * 0.8**3 * 0.2 + 0.5 * 0.5**4)   # direct Bayes check
print(f"direct Bayes = {direct:.4f}")
approx(odds, 1.6384, 1e-12); approx(p_biased, direct, 1e-12)
assert abs(p_biased - 0.6210) < 5e-5

per-flip Bayes factors = [1.6 1.6 0.4 1.6]
posterior odds = 1.6384   P(biased) = 0.6210
log-odds = 0.4937   sigma(log-odds) = 0.6210
direct Bayes = 0.6210


### Problem L1.6 — Gambler's Ruin via First-Step Conditioning

**Statement.** A gambler starts with 1 unit and bets fair 1-unit coin flips, stopping at 0 (ruin) or 3 (target). Using first-step analysis (conditioning on the first flip), compute the probability of reaching 3 before 0.

**Intuition.** Conditioning on the very first flip expresses the answer at fortune $i$ in terms of the answers at $i \pm 1$, turning a random walk into two linear equations.

**Solution.**

*Step 1.* Let $h_i$ be the probability of reaching $3$ before $0$ from fortune $i$, with $h_0 = 0$ and $h_3 = 1$. Condition on the first flip (a two-set partition):

$$
h_i = \tfrac12 h_{i+1} + \tfrac12 h_{i-1}, \qquad i = 1, 2 .
$$

*Step 2.* Solve the two equations by substitution. From $i = 1$, $h_1 = \tfrac12 h_2$. From $i = 2$, $h_2 = \tfrac12 + \tfrac12 h_1 = \tfrac12 + \tfrac14 h_2$, hence $h_2 = \tfrac23$ and $h_1 = \tfrac13$.

*Step 3.* Note the structure: rewritten as $h_{i+1} - 2h_i + h_{i-1} = 0$, the recursion says the second difference vanishes, so any solution is affine, $h_i = a + bi$; the boundary values then give $h_i = i/3$ directly, in agreement with Step 2.

$$
\boxed{P(\text{reach } 3 \text{ before } 0 \mid \text{start at } 1) = \frac{1}{3}}
$$

**Key takeaway.** First-step conditioning converts a dynamic random process into a linear recurrence — the prototype of Markov-chain absorption analysis.

In [11]:
# solve the linear system for h1, h2, then simulate the walk
Amat = np.array([[1.0, -0.5], [-0.5, 1.0]])
bvec = np.array([0.0, 0.5])                 # h1 = .5 h2 + .5 h0 ; h2 = .5 h3 + .5 h1
h = np.linalg.solve(Amat, bvec)
print(f"h1 = {h[0]:.6f}  h2 = {h[1]:.6f}   (affine solution i/3 = {1/3:.6f}, {2/3:.6f})")

reps = 200_000                              # vectorized walk with absorbing barriers 0 and 3
pos = np.ones(reps, dtype=int)
for _ in range(2000):
    active = (pos > 0) & (pos < 3)
    if not active.any():
        break
    pos[active] += rng.integers(0, 2, int(active.sum())) * 2 - 1
hits = (pos == 3).mean()
print(f"simulated P(hit 3 first) = {hits:.4f}   (all walks absorbed: {np.all((pos == 0) | (pos == 3))})")
approx(h[0], 1/3, 1e-12); approx(h[1], 2/3, 1e-12)
assert abs(hits - 1/3) < 0.01

h1 = 0.333333  h2 = 0.666667   (affine solution i/3 = 0.333333, 0.666667)
simulated P(hit 3 first) = 0.3329   (all walks absorbed: True)


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Naive Bayes Spam Filter

**Statement.** A spam filter uses two words with conditional probabilities $P(\text{"free"} \mid S) = 0.6$, $P(\text{"free"} \mid S^c) = 0.05$, $P(\text{"meeting"} \mid S) = 0.02$, $P(\text{"meeting"} \mid S^c) = 0.3$, and prior $P(S) = 0.4$. Assuming conditional independence of words given the class, classify an email containing both "free" and "meeting".

**Intuition.** Each word contributes its own likelihood ratio; here the two words pull in opposite directions, so the verdict is decided by which ratio is larger.

**Solution.**

*Step 1.* Multiply the per-word likelihoods, which is exactly the naive (conditional independence) assumption:

$$
P(\text{both} \mid S) = 0.6 \times 0.02 = 0.012, \qquad P(\text{both} \mid S^c) = 0.05 \times 0.3 = 0.015 .
$$

*Step 2.* Weight by the prior to get unnormalized posteriors: spam $0.012 \times 0.4 = 0.0048$, ham $0.015 \times 0.6 = 0.0090$.

*Step 3.* Normalize:

$$
P(S \mid \text{both}) = \frac{0.0048}{0.0048 + 0.0090} = \frac{0.0048}{0.0138} \approx 0.3478 ,
$$

so the email is classified as ham.

*Step 4.* Check in odds form: "free" has Bayes factor $12$ for spam, "meeting" has $0.02/0.3 = 1/15$, and prior odds are $0.4/0.6 = 2/3$, so the posterior odds are $\frac23 \times 12 \times \frac{1}{15} = 0.5333$, matching $0.3478/0.6522$.

$$
\boxed{P(\text{spam} \mid \text{"free", "meeting"}) \approx 0.3478 \implies \text{ham}}
$$

**Key takeaway.** Naive Bayes multiplies per-feature likelihood ratios — an assumption of conditional independence that is often wrong yet remarkably useful.

In [12]:
lik_spam = 0.6 * 0.02
lik_ham = 0.05 * 0.3
prior_s = 0.4
post_s = lik_spam * prior_s / (lik_spam * prior_s + lik_ham * (1 - prior_s))
odds = (prior_s / (1 - prior_s)) * (0.6/0.05) * (0.02/0.3)
print(f"P(both|S) = {lik_spam:.4f}   P(both|S^c) = {lik_ham:.4f}")
print(f"P(S | both) = {post_s:.4f}   posterior odds = {odds:.4f} -> {odds/(1+odds):.4f}")
approx(post_s, odds / (1 + odds), 1e-12)
assert abs(post_s - 0.3478) < 5e-5 and post_s < 0.5

P(both|S) = 0.0120   P(both|S^c) = 0.0150
P(S | both) = 0.3478   posterior odds = 0.5333 -> 0.3478


### Problem L2.2 — Precision, Recall, and Prevalence

**Statement.** A fraud detector has recall $P(\hat{F} \mid F) = 0.90$ and false-alarm rate $P(\hat{F} \mid F^c) = 0.01$, with fraud prevalence $P(F) = 0.002$. Compute the precision $P(F \mid \hat{F})$, and the prevalence needed for precision $0.5$.

**Intuition.** Precision is a posterior, so it is a competition between $0.9 \times$ a tiny prior and $0.01 \times$ an enormous one; the enormous one wins.

**Solution.**

*Step 1.* Apply Bayes with the partition $\{F, F^c\}$:

$$
P(F \mid \hat{F}) = \frac{0.90 \times 0.002}{0.90 \times 0.002 + 0.01 \times 0.998} = \frac{0.0018}{0.0018 + 0.00998} = \frac{0.0018}{0.01178} \approx 0.1528 .
$$

*Step 2.* Only $15.3\%$ of flagged transactions are fraudulent despite a strong detector — the false alarms come from a base that is $499$ times larger.

*Step 3.* Precision $0.5$ means true positives equal false positives:

$$
0.90\,\pi = 0.01\,(1 - \pi) \implies \pi = \frac{0.01}{0.91} \approx 0.010989 ,
$$

about $1.1\%$, five and a half times the current prevalence. Alternatively hold the prevalence and cut the false-alarm rate to $0.90 \times 0.002/0.998 \approx 0.0018$.

$$
\boxed{P(F \mid \hat{F}) \approx 0.1528; \text{ precision } 0.5 \text{ needs prevalence } \approx 1.1\%}
$$

**Key takeaway.** Precision depends on prevalence, not just on the classifier; rare-event detection is prior-limited, not model-limited.

In [13]:
recall, far, prev = 0.90, 0.01, 0.002
precision = recall * prev / (recall * prev + far * (1 - prev))
pi_star = far / (recall + far)
far_star = recall * prev / (1 - prev)
print(f"precision at prevalence {prev} = {precision:.4f}")
print(f"prevalence for precision 0.5   = {pi_star:.6f}")
print(f"check: precision there         = "
      f"{recall * pi_star / (recall * pi_star + far * (1 - pi_star)):.4f}")
print(f"false-alarm rate for precision 0.5 at the original prevalence = {far_star:.6f}")
approx(recall * pi_star / (recall * pi_star + far * (1 - pi_star)), 0.5, 1e-12)
assert abs(precision - 0.1528) < 5e-5 and abs(pi_star - 0.010989) < 1e-6

precision at prevalence 0.002 = 0.1528
prevalence for precision 0.5   = 0.010989
check: precision there         = 0.5000
false-alarm rate for precision 0.5 at the original prevalence = 0.001804


### Problem L2.3 — Chain Rule and Language-Model Perplexity

**Statement.** An autoregressive model assigns $P(w_1) = 0.2$, $P(w_2 \mid w_1) = 0.5$, $P(w_3 \mid w_1, w_2) = 0.25$ to a 3-token sentence. (a) Compute the joint probability. (b) Compute the per-token perplexity $\operatorname{PPL} = P(w_{1:3})^{-1/3}$. (c) Show that PPL is the exponential of the average per-token cross-entropy.

**Intuition.** The chain rule turns the sentence probability into a product, so its geometric mean per token is the model's effective branching factor.

**Solution.**

*Step 1.* Chain rule: $P(w_{1:3}) = 0.2 \times 0.5 \times 0.25 = 0.025$.

*Step 2.* Perplexity: $\operatorname{PPL} = 0.025^{-1/3} = 40^{1/3} \approx 3.4200$.

*Step 3.* Average negative log-probability per token:

$$
H = -\tfrac13 \ln P(w_{1:3}) = -\tfrac13\left(\ln 0.2 + \ln 0.5 + \ln 0.25\right) \approx 1.2296 .
$$

*Step 4.* Exponentiate: $e^{H} = e^{1.2296} \approx 3.4200 = \operatorname{PPL}$, because $\exp(-\frac1T \sum_t \ln P_t) = \left(\prod_t P_t\right)^{-1/T}$ identically.

$$
\boxed{P(w_{1:3}) = 0.025, \qquad \operatorname{PPL} = 40^{1/3} \approx 3.4200 = e^{H}}
$$

**Key takeaway.** The chain rule makes sequence likelihood a product of conditionals; perplexity is its inverse geometric mean — the standard LLM evaluation metric.

In [14]:
cond = np.array([0.2, 0.5, 0.25])
joint = float(np.prod(cond))
ppl = joint ** (-1/3)
H = float(-np.mean(np.log(cond)))
print(f"joint = {joint:.4f}   PPL = {ppl:.4f}   40^(1/3) = {40 ** (1/3):.4f}")
print(f"mean cross-entropy H = {H:.4f} nats   exp(H) = {np.exp(H):.4f}")
approx(joint, 0.025, 1e-15); approx(ppl, 40 ** (1/3), 1e-12); approx(np.exp(H), ppl, 1e-12)

joint = 0.0250   PPL = 3.4200   40^(1/3) = 3.4200
mean cross-entropy H = 1.2296 nats   exp(H) = 3.4200


np.float64(3.4199518933533932)

### Problem L2.4 — Explaining Away in a Bayesian Network

**Statement.** Grades depend on talent $T$ and effort $E$, with independent priors $P(T) = 0.2$, $P(E) = 0.5$. An A is earned with probability $0.9$ if both hold, $0.5$ if exactly one holds, $0.1$ if neither. Given a student got an A, compute $P(T \mid A)$ and $P(T \mid A, E)$, and interpret the difference.

**Intuition.** Talent and effort compete to explain the same A; once effort is known to be present, less of the A needs to be attributed to talent.

**Solution.**

*Step 1.* Enumerate the four independent cause combinations and their contribution to $P(A)$:

$$
P(A) = 0.9(0.2)(0.5) + 0.5(0.2)(0.5) + 0.5(0.8)(0.5) + 0.1(0.8)(0.5) = 0.09 + 0.05 + 0.20 + 0.04 = 0.38 .
$$

*Step 2.* Collect the terms containing $T$: $P(T \cap A) = 0.09 + 0.05 = 0.14$, so

$$
P(T \mid A) = \frac{0.14}{0.38} \approx 0.3684 .
$$

*Step 3.* Condition on $E$ as well, using $P(A \mid T, E) = 0.9$ and $P(A \mid T^c, E) = 0.5$:

$$
P(T \mid A, E) = \frac{0.9 \times 0.2}{0.9 \times 0.2 + 0.5 \times 0.8} = \frac{0.18}{0.58} \approx 0.3103 .
$$

*Step 4.* Interpret: learning that the student worked hard *lowers* the talent posterior from $0.3684$ to $0.3103$, although $T$ and $E$ are independent a priori.

$$
\boxed{P(T \mid A) \approx 0.3684 \gt P(T \mid A, E) \approx 0.3103}
$$

**Key takeaway.** Conditioning on a common effect (a collider) induces negative dependence between its independent causes — the signature pattern of graphical models and a mechanism behind selection bias.

In [15]:
pT, pE = 0.2, 0.5
cpt = {(1, 1): 0.9, (1, 0): 0.5, (0, 1): 0.5, (0, 0): 0.1}     # P(A | T, E)
cells = {(t, e): (pT if t else 1 - pT) * (pE if e else 1 - pE) for t in (0, 1) for e in (0, 1)}
PA = sum(cpt[k] * v for k, v in cells.items())
PTA = sum(cpt[k] * v for k, v in cells.items() if k[0] == 1)
PTAE = sum(cpt[k] * v for k, v in cells.items() if k == (1, 1))
PAE = sum(cpt[k] * v for k, v in cells.items() if k[1] == 1)
print(f"P(A) = {PA:.4f}   P(T|A) = {PTA/PA:.4f}   P(T|A,E) = {PTAE/PAE:.4f}")
approx(PA, 0.38, 1e-12); approx(PTA/PA, 0.14/0.38, 1e-12); approx(PTAE/PAE, 0.18/0.58, 1e-12)
assert PTAE/PAE < PTA/PA

P(A) = 0.3800   P(T|A) = 0.3684   P(T|A,E) = 0.3103


### Problem L2.5 — Canonical Ensemble by Conditioning (Physics)

**Statement.** A two-state system with energies $0$ and $\epsilon$ is in contact with a reservoir having $\Omega_R(E)$ microstates at energy $E$, with $\ln \Omega_R(E_{\text{tot}} - \epsilon) \approx \ln \Omega_R(E_{\text{tot}}) - \beta\epsilon$ (first-order Taylor, $\beta = 1/k_B T$). Assuming all joint microstates of fixed total energy are equally likely, derive the probability that the system occupies the energy-$\epsilon$ state.

**Intuition.** The system's state is only as likely as the number of reservoir configurations that can pay for it, and the reservoir's microstate count falls off exponentially in the energy it must give up.

**Solution.**

*Step 1.* The joint (microcanonical) distribution is uniform over microstates of total energy $E_{\text{tot}}$. Conditioning on the system's state, the number of compatible joint microstates is proportional to the reservoir count:

$$
P(\text{system at } \epsilon) \propto \Omega_R(E_{\text{tot}} - \epsilon), \qquad P(\text{system at } 0) \propto \Omega_R(E_{\text{tot}}) .
$$

*Step 2.* Take the ratio and use the Taylor expansion of the log-count:

$$
\frac{P(\epsilon)}{P(0)} = \frac{\Omega_R(E_{\text{tot}} - \epsilon)}{\Omega_R(E_{\text{tot}})} = e^{-\beta\epsilon} .
$$

*Step 3.* Renormalize over the two states:

$$
P(\epsilon) = \frac{e^{-\beta\epsilon}}{1 + e^{-\beta\epsilon}} = \sigma(-\beta\epsilon),
$$

the Boltzmann distribution — literally a logistic sigmoid of the energy gap.

$$
\boxed{P(\epsilon) = \frac{e^{-\beta\epsilon}}{1 + e^{-\beta\epsilon}} = \sigma(-\beta\epsilon)}
$$

**Key takeaway.** The Boltzmann factor is a conditional probability: the uniform measure on the joint system, conditioned on what the reservoir can absorb. Softmax layers reuse exactly this structure.

In [16]:
# concrete reservoir: an Einstein solid of N oscillators holding q quanta, epsilon = 1 quantum
from math import lgamma
N, q = 4000, 12000
ln_omega = lambda k: lgamma(k + N) - lgamma(k + 1) - lgamma(N)
exact_ratio = exp(ln_omega(q - 1) - ln_omega(q))          # Omega_R(E-eps)/Omega_R(E)
beta_eps = digamma(q + N) - digamma(q + 1)                # d(ln Omega)/dE at E_tot, times eps=1
print(f"exact count ratio      = {exact_ratio:.8f}")
print(f"Boltzmann exp(-beta e) = {exp(-beta_eps):.8f}   (Taylor step, error "
      f"{abs(exact_ratio - exp(-beta_eps)):.2e})")
P_eps_exact = exact_ratio / (1 + exact_ratio)
P_eps_boltz = 1 / (1 + exp(beta_eps))                     # sigma(-beta eps)
print(f"P(eps) exact = {P_eps_exact:.8f}   sigma(-beta eps) = {P_eps_boltz:.8f}")
assert abs(P_eps_exact - P_eps_boltz) < 1e-4

exact count ratio      = 0.75004688
Boltzmann exp(-beta e) = 0.75005469   (Taylor step, error 7.81e-06)
P(eps) exact = 0.42858674   sigma(-beta eps) = 0.42858929


### Problem L2.6 — Sensor Fusion with Two Independent Detectors (Physics)

**Statement.** A target is present with prior $P(T) = 0.1$. Two detectors, conditionally independent given target status, each report "detect" with $P(D_i \mid T) = 0.8$ and $P(D_i \mid T^c) = 0.1$. Compute $P(T \mid D_1, D_2)$ when both detect, and compare with a single detection.

**Intuition.** Conditional independence lets each detector contribute its own likelihood ratio, so two mediocre sensors compound into one confident one.

**Solution.**

*Step 1.* Convert the prior to odds: $O(T) = 0.1/0.9 = 1/9$.

*Step 2.* Each detection carries Bayes factor $0.8/0.1 = 8$; conditional independence means the factors multiply:

$$
\text{posterior odds} = \frac19 \times 8 \times 8 = \frac{64}{9} \approx 7.111 \implies P(T \mid D_1, D_2) = \frac{64}{73} \approx 0.8767 .
$$

*Step 3.* One detection only: odds $8/9$, so $P(T \mid D_1) = 8/17 \approx 0.4706$.

*Step 4.* Verify directly, without odds:

$$
P(T \mid D_1, D_2) = \frac{0.8^2 \times 0.1}{0.8^2 \times 0.1 + 0.1^2 \times 0.9} = \frac{0.064}{0.073} = \frac{64}{73}.
$$

$$
\boxed{P(T \mid D_1, D_2) = \frac{64}{73} \approx 0.8767 \quad (\text{versus } 0.4706 \text{ for one detector})}
$$

**Key takeaway.** Conditionally independent evidence multiplies Bayes factors — the arithmetic behind sensor fusion and ensemble methods.

In [17]:
prior, s, f = 0.1, 0.8, 0.1
odds1 = (prior / (1 - prior)) * (s / f)
odds2 = (prior / (1 - prior)) * (s / f) ** 2
direct2 = s**2 * prior / (s**2 * prior + f**2 * (1 - prior))
print(f"one detection : odds {odds1:.4f} -> P = {odds1/(1+odds1):.4f}   (8/17 = {8/17:.4f})")
print(f"two detections: odds {odds2:.4f} -> P = {odds2/(1+odds2):.4f}   (64/73 = {64/73:.4f})")
print(f"direct Bayes  : {direct2:.4f}")
approx(odds2/(1+odds2), 64/73, 1e-12); approx(direct2, 64/73, 1e-12)
approx(odds1/(1+odds1), 8/17, 1e-12)

one detection : odds 0.8889 -> P = 0.4706   (8/17 = 0.4706)
two detections: odds 7.1111 -> P = 0.8767   (64/73 = 0.8767)
direct Bayes  : 0.8767


0.4705882352941177

## L3 — Challenge Proofs

### Problem L3.1 — The Two-Child Paradox (Information Mechanisms Matter)

**Statement.** A family has two children, each independently boy/girl with probability $1/2$. Compute $P(\text{two boys})$ given: (a) the older child is a boy; (b) at least one child is a boy; (c) a randomly chosen one of the two children is seen to be a boy.

**Intuition.** The three phrasings condition on three different events, and (c) is not a fact about the family at all but the outcome of a sampling procedure.

**Solution.**

*Step 1.* Fix the uniform sample space $\{BB, BG, GB, GG\}$ with the older child listed first.

*Step 2.* (a) The condition is $\{BB, BG\}$, of probability $1/2$, so

$$
P(BB \mid \text{older is a boy}) = \frac{1/4}{1/2} = \frac12 .
$$

*Step 3.* (b) The condition is $\{BB, BG, GB\}$, of probability $3/4$, so

$$
P(BB \mid \text{at least one boy}) = \frac{1/4}{3/4} = \frac13 .
$$

*Step 4.* (c) Model the mechanism: pick one child uniformly and observe the sex. The likelihoods are $P(\text{see B} \mid BB) = 1$, $P(\text{see B} \mid BG) = P(\text{see B} \mid GB) = \frac12$, $P(\text{see B} \mid GG) = 0$, so by Bayes

$$
P(BB \mid \text{see a boy}) = \frac{1 \cdot \frac14}{1 \cdot \frac14 + \frac12 \cdot \frac14 + \frac12 \cdot \frac14 + 0} = \frac{1/4}{1/2} = \frac12 .
$$

*Step 5.* Compare (b) and (c): "a boy exists" and "a boy was seen" are different events — the second is generated by a sampling rule that is twice as likely to fire in a two-boy family.

$$
\boxed{\text{(a) } \tfrac12, \quad \text{(b) } \tfrac13, \quad \text{(c) } \tfrac12}
$$

**Key takeaway.** Correct conditioning requires modelling how the information reached you — the likelihood of the observation process, not just the truth of the revealed statement.

In [18]:
n = 400_000
kids = rng.integers(0, 2, size=(n, 2))            # 1 = boy
both = kids.sum(axis=1) == 2
older = kids[:, 0] == 1
atleast = kids.sum(axis=1) >= 1
seen_idx = rng.integers(0, 2, size=n)             # the sampling mechanism of case (c)
seen_boy = kids[np.arange(n), seen_idx] == 1
print(f"(a) P(BB | older boy)   = {both[older].mean():.4f}   exact 0.5")
print(f"(b) P(BB | >=1 boy)     = {both[atleast].mean():.4f}   exact {1/3:.4f}")
print(f"(c) P(BB | saw a boy)   = {both[seen_boy].mean():.4f}   exact 0.5")
assert abs(both[older].mean() - 0.5) < 0.01
assert abs(both[atleast].mean() - 1/3) < 0.01
assert abs(both[seen_boy].mean() - 0.5) < 0.01

(a) P(BB | older boy)   = 0.5004   exact 0.5
(b) P(BB | >=1 boy)     = 0.3339   exact 0.3333
(c) P(BB | saw a boy)   = 0.5005   exact 0.5


### Problem L3.2 — Simpson's Paradox Constructed and Resolved

**Statement.** Treatment A has higher success rates than B in *both* mild and severe cases, yet a lower overall rate. Construct explicit numbers exhibiting this and explain via the law of total probability.

**Intuition.** Aggregate rates are weighted averages, and if the two treatments face very different case mixes, the weights can overturn the comparison.

**Solution.**

*Step 1.* Construct the table (successes / patients):

| Group | Treatment A | Treatment B |
|---|---|---|
| Mild | $81/87 \approx 0.931$ | $234/270 \approx 0.867$ |
| Severe | $192/263 \approx 0.730$ | $55/80 \approx 0.688$ |
| **Overall** | $273/350 = 0.780$ | $289/350 \approx 0.826$ |

*Step 2.* Verify the reversal: A wins in both strata ($0.931 \gt 0.867$ and $0.730 \gt 0.688$) yet loses overall ($0.780 \lt 0.826$).

*Step 3.* Expand each overall rate by the law of total probability over severity:

$$
P(S \mid A) = P(S \mid A, M)\,P(M \mid A) + P(S \mid A, \Sigma)\,P(\Sigma \mid A),
$$

and note the weights differ drastically: A treats mostly severe cases ($263/350 \approx 75\%$), B mostly mild ($270/350 \approx 77\%$).

*Step 4.* Diagnose: severity is a confounder, correlated with both the treatment assignment and the outcome, so the aggregate comparison mixes incomparable case mixes. Equalizing the weights (standardization) restores A's advantage.

$$
\boxed{\text{A better in both strata, worse in aggregate — the weights } P(\text{stratum} \mid \text{treatment}) \text{ differ}}
$$

**Key takeaway.** Marginal comparisons can invert conditional ones when the conditioning weights differ; always ask what the mixture weights are before averaging rates.

In [19]:
tab = {"A": {"mild": (81, 87), "severe": (192, 263)},
       "B": {"mild": (234, 270), "severe": (55, 80)}}
rates, overall = {}, {}
for t, strata in tab.items():
    rates[t] = {k: s / n for k, (s, n) in strata.items()}
    s_tot = sum(s for s, _ in strata.values()); n_tot = sum(n for _, n in strata.values())
    overall[t] = s_tot / n_tot
    print(f"{t}: mild {rates[t]['mild']:.3f}  severe {rates[t]['severe']:.3f}  "
          f"overall {s_tot}/{n_tot} = {overall[t]:.3f}")
# standardized comparison: same case mix (half mild, half severe) for both treatments
std = {t: 0.5 * rates[t]["mild"] + 0.5 * rates[t]["severe"] for t in tab}
print(f"standardized to a 50/50 case mix: A {std['A']:.3f}  B {std['B']:.3f}")
assert rates["A"]["mild"] > rates["B"]["mild"] and rates["A"]["severe"] > rates["B"]["severe"]
assert overall["A"] < overall["B"] and std["A"] > std["B"]

A: mild 0.931  severe 0.730  overall 273/350 = 0.780
B: mild 0.867  severe 0.688  overall 289/350 = 0.826
standardized to a 50/50 case mix: A 0.831  B 0.777


### Problem L3.3 — Borel's Paradox: Conditioning on a Zero-Probability Event

**Statement.** Explain why "conditioning on $B$" is ill-defined when $P(B) = 0$, illustrate with the two natural answers for a uniform point on a sphere conditioned on lying on a great circle through the poles, and state the resolution used in modern probability.

**Intuition.** A null event has to be reached as a limit of fat events, and different ways of fattening it concentrate mass differently along it.

**Solution.**

*Step 1.* The ratio definition collapses: $P(A \mid B) = P(A \cap B)/P(B)$ is $0/0$ when $P(B) = 0$. The natural repair is a limit $\lim_{\varepsilon \to 0} P(A \mid B_\varepsilon)$ over events $B_\varepsilon \downarrow B$ of positive probability.

*Step 2.* Fix the conditioning set: the great circle through the poles at longitudes $\varphi_0$ and $\varphi_0 + \pi$ — a full circle, formed by two antipodal meridians (a single meridian is only half of it).

*Step 3.* Fatten it as a thin **lune of longitude**, $\lvert \varphi - \varphi_0 \rvert \lt \varepsilon$ or $\lvert \varphi - \varphi_0 - \pi \rvert \lt \varepsilon$. Lunes are widest at the equator, so the limiting density on the circle is proportional to $\cos(\text{latitude})$, and the mean absolute latitude is

$$
\frac{\int_0^{\pi/2} \lambda \cos\lambda \, d\lambda}{\int_0^{\pi/2}\cos\lambda\,d\lambda} = \frac{\pi}{2} - 1 \approx 0.5708\ \text{rad} \approx 32.70^\circ .
$$

*Step 4.* Fatten it instead as a thin **band of constant angular width** around the same circle. Rotating the sphere about the axis perpendicular to the circle's plane maps the band to itself, so by Archimedes' theorem the limiting density is uniform in arc length, giving mean absolute latitude $45^\circ$.

*Step 5.* Both limits are internally consistent yet disagree: this is Borel's paradox. Kolmogorov's resolution is that conditional probability is defined relative to a $\sigma$-algebra — equivalently, relative to an entire partition into null events, via $E[\mathbf{1}_A \mid \mathcal{G}]$ or a regular conditional distribution. Choosing the partition is choosing the limiting mechanism, and the paradox evaporates once it is stated.

$$
\boxed{\text{Conditioning on a null event is partition-dependent: } 32.70^\circ \text{ versus } 45^\circ \text{ for the same circle}}
$$

**Key takeaway.** For continuous variables, $p(y \mid X = x)$ is well defined only relative to how "$X = x$" is embedded in a family of events — the subtlety behind regular conditional distributions.

In [20]:
# simulate both fattening schemes for the same great circle (the one in the x-z plane)
n = 3_000_000
z = rng.uniform(-1, 1, n)                       # uniform on the sphere: z is uniform
phi = rng.uniform(0, 2 * np.pi, n)
y = np.sqrt(1 - z**2) * np.sin(phi)
lat = np.degrees(np.arcsin(z))

lune = np.abs(np.sin(phi)) < 0.02               # thin lune: longitude near phi_0 or phi_0 + pi
band = np.abs(y) < 0.02                        # thin band of constant width around the circle
print(f"lune of longitude : kept {lune.sum():6d}   mean |latitude| = "
      f"{np.abs(lat[lune]).mean():.2f} deg   predicted {np.degrees(np.pi/2 - 1):.2f}")
print(f"constant-width band: kept {band.sum():6d}   mean |latitude| = "
      f"{np.abs(lat[band]).mean():.2f} deg   predicted 45.00")
assert abs(np.abs(lat[lune]).mean() - np.degrees(np.pi/2 - 1)) < 0.5
assert abs(np.abs(lat[band]).mean() - 45.0) < 0.5

lune of longitude : kept  38127   mean |latitude| = 32.73 deg   predicted 32.70


constant-width band: kept  59795   mean |latitude| = 44.93 deg   predicted 45.00


### Problem L3.4 — Optimality of Bayesian Updating (Dutch Book)

**Statement.** Show that an agent whose conditional beliefs violate $P(A \mid B) = P(A \cap B)/P(B)$ accepts a finite set of bets guaranteeing a sure loss (a Dutch book), and conclude that the ratio rule is the unique coherent conditional price.

**Intuition.** A called-off bet on $A$ given $B$ can be replicated by two ordinary bets, so its price is forced; any other price is an arbitrage.

**Solution.**

*Step 1.* Model the instruments. A *conditional bet on $A$ given $B$* at rate $q$ pays $1 - q$ on $A \cap B$, pays $-q$ on $A^c \cap B$, and is called off (payoff $0$) on $B^c$. An ordinary bet at rate $r$ on an event pays $1 - r$ if it occurs and $-r$ otherwise.

*Step 2.* Let the agent post $q = P^*(A \mid B)$, $r = P(A \cap B)$ and $s = P(B)$ with $q \ne r/s$, and let the bookmaker take stakes $\alpha$, $\beta$, $\gamma$ on the three instruments. The agent's net payoff in the three exhaustive cases is

- on $A \cap B$: $\alpha(1-q) + \beta(1-r) + \gamma(1-s)$;
- on $A^c \cap B$: $-\alpha q - \beta r + \gamma(1-s)$;
- on $B^c$: $-\beta r - \gamma s$.

*Step 3.* Choose $\beta = -\alpha$ and $\gamma = \alpha q$. Each case collapses:

- on $A \cap B$: $\alpha(1-q) - \alpha(1-r) + \alpha q(1-s) = \alpha(r - qs)$;
- on $A^c \cap B$: $-\alpha q + \alpha r + \alpha q(1-s) = \alpha(r - qs)$;
- on $B^c$: $\alpha r - \alpha q s = \alpha(r - qs)$.

*Step 4.* All three payoffs are the same number $\alpha(r - qs)$, a *riskless* position. If $q \ne r/s$ then $r - qs \ne 0$, and the bookmaker picks the sign of $\alpha$ making $\alpha(r-qs) \lt 0$: the agent loses the same positive amount in every state of the world.

*Step 5.* Coherence therefore forces $qs = r$, i.e. $P^*(A \mid B) = P(A \cap B)/P(B)$, which is the definition of conditional probability; the update rule is not a convention but the unique arbitrage-free price. $\blacksquare$

$$
\boxed{\text{Any conditional price other than } P(A \mid B) = P(A \cap B)/P(B) \text{ is exploitable for sure loss}}
$$

**Key takeaway.** Bayes' rule is the unique betting-coherent update — the Ramsey/de Finetti foundation beneath Bayesian machine learning.

In [21]:
# verify the payoff collapse symbolically-in-numbers for an incoherent agent
r, s = 0.12, 0.40                      # P(A and B), P(B)  ->  coherent q would be 0.30
for q in (0.30, 0.45):
    alpha = 1.0 if r - q * s < 0 else -1.0        # bookmaker picks the profitable sign
    beta, gamma = -alpha, alpha * q
    pay_AB = alpha * (1 - q) + beta * (1 - r) + gamma * (1 - s)
    pay_AcB = -alpha * q - beta * r + gamma * (1 - s)
    pay_Bc = -beta * r - gamma * s
    print(f"q = {q:.2f}: payoffs = ({pay_AB:+.4f}, {pay_AcB:+.4f}, {pay_Bc:+.4f})"
          f"   common value alpha*(r - q*s) = {alpha * (r - q * s):+.4f}")
    approx(pay_AB, pay_AcB, 1e-12); approx(pay_AB, pay_Bc, 1e-12)
    approx(pay_AB, alpha * (r - q * s), 1e-12)
print("coherent q = r/s =", r / s, "gives payoff 0 in every state; any other q is a sure loss")

q = 0.30: payoffs = (+0.0000, +0.0000, +0.0000)   common value alpha*(r - q*s) = -0.0000
q = 0.45: payoffs = (-0.0600, -0.0600, -0.0600)   common value alpha*(r - q*s) = -0.0600
coherent q = r/s = 0.3 gives payoff 0 in every state; any other q is a sure loss
